# Fourier-feature CFM for an eroded mountain landscape

Fit the hydraulic-erosion heightmap saved in `C:/Users/shich/Src/smooth_chaos/noise.ipynb` using **one hidden ReLU layer** conditioned on Fourier coordinate features. The target is the saved `eroded_height` surface. Its input field is reconstructed as `eroded_height - height_diff` from the two outputs of the same source cell. Both are bundled in `_shared/data/noise_hydraulic_terrain.npz`; there is no external runtime dependency. `TerrainConfig(source="reference", resolution=128)` loads these heightmaps directly. The `Saw`, `MountainGen`, and `hydraulic_erosion` definitions are also available in `_shared/mountains.py` for `source="generated"` targets.

The coordinate embedding contains normalized raw `(x, y)` plus sine/cosine bands at frequencies **1, 2, 4, 8, 16, 32** along four directions. Only coordinates receive Fourier features. The noisy height is concatenated to this embedding before `Linear -> ReLU -> Linear`; Fourier projections are fixed buffers, not trainable parameters.

**Training:** standardize the eroded height field, sample mini-batches of grid coordinates, draw `t ~ Uniform(0, 1)` and `z ~ Normal(0, 1)`, construct `s_t=(1-t)z+t*h_normalized`, and minimize `MSE(residual_hat, h_normalized-s_t)`. Time is not a network input. Coordinate features are cached because their frequencies are fixed.

**Inference:** use **64 Euler steps** with velocity `residual_hat/(1-t)`, starting from a zero normalized-height state at every coordinate. Convert predictions back to the original height units. The sampler never receives target heights. Coordinate features are computed once per inference chunk and reused across Euler steps.

The live renderer evaluates a 96x96 preview and shows height MSE, target and prediction as 3D surfaces, and a height-error map. After training, evaluate the entire terrain grid and compare both 3D surfaces with shared height/color scales. Grid MSE measures reconstruction of the generated heightmap, not a held-out test set. Weights, fixed Fourier buffers, target terrain, normalization, and configuration are saved together for repeatable inference.

In [1]:
from pathlib import Path
import sys

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import numpy as np
import torch
from IPython.display import display, Markdown
from _shared.mountains import TerrainConfig, generate_terrain
from _shared.fourier_mountain import (
    MountainFitConfig,
    MountainExperiment,
    surface_comparison,
    load_mountain_checkpoint,
)
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.visualization import train_live

torch.set_num_threads(1)

In [2]:
terrain_config = TerrainConfig(
    source="reference",
    resolution=128,
)
fit_config = MountainFitConfig(
    hidden_size=512,
    frequencies=(1, 2, 4, 8, 16, 32),
    directions=4,
    total_steps=32_000,
    batch_size=1024,
    learning_rate=0.001,
    seed=42,
    preview_resolution=96,
    inference_batch_size=2048,
    device="cpu",  # Set to "cuda" when using a CUDA-enabled PyTorch installation.
)
flow_config = FlowMatchingConfig(
    train_sampling_steps=None,
    eval_sampling_steps=64,
)
target_fps = 0.5
output_dir = project_root / "notebooks" / "outputs"
checkpoint_path = output_dir / "fourier_mountain_cfm.pt"

In [3]:
terrain = generate_terrain(terrain_config)
figure = surface_comparison(terrain)
figure.show(config={"responsive": True, "displayModeBar": False})

In [5]:
mountain_experiment = MountainExperiment(terrain, fit_config, flow_config)
parameter_count = sum(
    parameter.numel() for parameter in mountain_experiment.model.parameters()
)
display(
    Markdown(
        f"**Trainable parameters:** {parameter_count:,} | "
        f"**Coordinate features:** {mountain_experiment.model.features.output_size} | "
        f"**Training grid:** {terrain_config.resolution} x {terrain_config.resolution}"
    )
)
try:
    await train_live(mountain_experiment, target_fps=target_fps)
finally:
    # Save the current model and terrain even when training is interrupted.
    mountain_experiment.save(checkpoint_path)

**Trainable parameters:** 27,137 | **Coordinate features:** 50 | **Training grid:** 128 x 128

## Full-grid inference: 64 Euler steps

This cell uses the trained model and zero initial noise. It evaluates every terrain-grid point in chunks, reports MSE and MAE in original height units, and saves the target, prediction, error, and coordinate axes to an NPZ file.

In [6]:
prediction = mountain_experiment.predict_full_grid()
error = prediction - terrain.eroded_height
mse = float(np.mean(error**2))
mae = float(np.mean(np.abs(error)))
display(
    Markdown(
        f"**Grid MSE:** {mse:.8g} | **Grid MAE:** {mae:.8g} | "
        f"**Euler steps:** {mountain_experiment.flow_config.eval_sampling_steps} | "
        f"**Completed training steps:** {mountain_experiment.progress.step:,}"
    )
)
figure = surface_comparison(terrain, prediction)
figure.show(config={"responsive": True, "displayModeBar": False})
output_dir.mkdir(parents=True, exist_ok=True)
np.savez_compressed(
    output_dir / "fourier_mountain_cfm_prediction.npz",
    x=terrain.x,
    y=terrain.y,
    target=terrain.eroded_height,
    prediction=prediction,
    error=error,
)

**Grid MSE:** 2.7572203e-06 | **Grid MAE:** 0.001395799 | **Euler steps:** 64 | **Completed training steps:** 32,000

## Load saved weights for inference

In a fresh kernel, run imports and configuration, then this cell instead of terrain generation and training. The checkpoint includes the exact terrain and normalization. Run the full-grid inference cell afterward.

In [7]:
# Optional: restore the checkpoint without training or regenerating the terrain.
mountain_experiment = load_mountain_checkpoint(
    checkpoint_path, device=fit_config.device
)
terrain = mountain_experiment.terrain